# Cloud Matching — Full-Resolution Axial/Local, Energy Only (Kaggle T4×2 DDP)

이 노트북은 **native-128 Gaussian-mixture 전이 생성 → bottleneck 없는 full-resolution local/axial cross+self attention → full-band Energy distance만으로 implicit sampling 학습 → 분포·spatial energy·복원·rollout 분석**을 한 번에 수행합니다. Kaggle에서 Accelerator를 **GPU T4 x2**, Internet을 **On**으로 설정하세요. 결과는 `/kaggle/working/cloud_matching_div2k_fullres_axial_energy_only`에 저장합니다.

정답 cloud는 white/local/edge/high-frequency/low-rank/anisotropic 등 Gaussian family의 혼합으로 생성합니다. 모델의 위치별 noise energy `w`에는 CE나 strength label을 전혀 주지 않으며, sample 결과에 대한 Energy distance만으로 위치·강도·appearance를 모두 학습합니다. 로컬 실행 시에는 자동으로 작은 합성 이미지와 1-epoch smoke 설정을 사용합니다.

In [ ]:
from __future__ import annotations
import importlib, importlib.util, inspect, json, os, shutil, subprocess, sys, time
from pathlib import Path

IN_KAGGLE = Path('/kaggle').exists()
LOCAL_SMOKE = not IN_KAGGLE
DATASET_HANDLE = 'takihasan/div2k-dataset-for-super-resolution'
EPOCHS = 12
TRAIN_VARIANTS = 4
VAL_VARIANTS = 2
EVAL_SAMPLES = 16
FORCE_REBUILD_DATA = False
RESUME_IF_AVAILABLE = True
INIT_CHECKPOINT = ''  # Optional Kaggle path; ignored when latest.pt exists.

if IN_KAGGLE:
    REPO_DIR = Path('/kaggle/working/Cloud-Matching')
    if not (REPO_DIR / 'pyproject.toml').exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Ahnd6474/Cloud-Matching.git', str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
    OUTPUT_DIR = Path('/kaggle/working/cloud_matching_div2k_fullres_axial_energy_only')
    CACHE_DIR = Path('/kaggle/temp/cloud_matching_div2k_fullres_axial_energy_v4_128')
else:
    here = Path.cwd().resolve()
    REPO_DIR = here if (here / 'pyproject.toml').exists() else here.parent
    OUTPUT_DIR = REPO_DIR / 'runs/kaggle-notebook-fullres-energy-smoke'
    CACHE_DIR = REPO_DIR / 'runs/kaggle-notebook-fullres-energy-smoke-fixed'
    EPOCHS, TRAIN_VARIANTS, VAL_VARIANTS, EVAL_SAMPLES = 1, 1, 1, 8

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)
SRC_DIR = (REPO_DIR / 'src').resolve()
if not (SRC_DIR / 'stochastic_bridge/__init__.py').exists():
    raise FileNotFoundError(f'Cloud-Matching package source not found: {SRC_DIR}')
# A pip subprocess cannot refresh the already-running notebook kernel's .pth files.
# Put src on both this kernel's path and child-process PYTHONPATH explicitly.
src_text = str(SRC_DIR)
if src_text not in sys.path:
    sys.path.insert(0, src_text)
previous_pythonpath = os.environ.get('PYTHONPATH', '')
pythonpath_entries = [entry for entry in previous_pythonpath.split(os.pathsep) if entry]
if src_text not in pythonpath_entries:
    os.environ['PYTHONPATH'] = os.pathsep.join([src_text, *pythonpath_entries])
required = {'geomloss': 'geomloss', 'yaml': 'pyyaml', 'tqdm': 'tqdm'}
if IN_KAGGLE:
    required['kagglehub'] = 'kagglehub'
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
    importlib.invalidate_caches()
for module_name in [name for name in sys.modules if name == 'stochastic_bridge' or name.startswith('stochastic_bridge.')]:
    del sys.modules[module_name]
import stochastic_bridge
from stochastic_bridge.model import StochasticImageBridge as BootstrapBridge
from stochastic_bridge.config import ExperimentConfig
if 'return_noise_energy' not in inspect.signature(BootstrapBridge.forward).parameters or 'architecture' not in inspect.signature(BootstrapBridge).parameters or not hasattr(stochastic_bridge, 'FullBandEnergyCorrectionCloudLoss'):
    raise RuntimeError('Cloned repository is older than the full-resolution Energy-only notebook. Commit/push the current Cloud-Matching code, restart the Kaggle session, and run from the first cell.')
print({'kaggle': IN_KAGGLE, 'repo': str(REPO_DIR), 'package': stochastic_bridge.__file__, 'output': str(OUTPUT_DIR), 'cache': str(CACHE_DIR)})

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch.utils.data import DataLoader
from torchvision.utils import make_grid

SEED = 17
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print('torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available(), 'GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(i, p.name, f'{p.total_memory / 2**30:.1f} GiB')
if IN_KAGGLE and torch.cuda.device_count() < 2:
    raise RuntimeError('Kaggle Accelerator를 GPU T4 x2로 설정하세요.')

## 1. DIV2K 다운로드와 split 탐색
Kaggle에서는 `kagglehub`가 dataset을 Input cache에 붙입니다. HR train/validation 폴더를 이름과 이미지 수로 탐색하므로 업로더의 중첩 폴더 구조가 조금 달라도 동작합니다.

In [ ]:
IMAGE_SUFFIXES = {'.png', '.jpg', '.jpeg', '.webp', '.bmp'}

def image_files(root):
    return sorted(p for p in Path(root).rglob('*') if p.suffix.lower() in IMAGE_SUFFIXES)

def find_hr_split(root, split):
    candidates = {}
    for path in image_files(root):
        text = str(path.parent).lower()
        if split in text and ('hr' in text or 'high' in text) and 'lr' not in text and 'bicubic' not in text:
            candidates.setdefault(path.parent, 0)
            candidates[path.parent] += 1
    if not candidates:
        raise FileNotFoundError(f'Could not locate {split} HR images below {root}')
    return max(candidates, key=candidates.get)

def make_local_sources(root):
    for split, count in [('train', 12), ('valid', 4)]:
        folder = root / split / 'HR'; folder.mkdir(parents=True, exist_ok=True)
        for index in range(count):
            size = 96
            yy, xx = np.mgrid[:size, :size] / (size - 1)
            circle = ((xx - .5)**2 + (yy - .5)**2 < (.12 + .015 * index)**2).astype(float)
            image = np.stack([(np.sin((index % 5 + 2) * np.pi * xx) + 1)/2, yy, circle], -1)
            Image.fromarray((image * 255).astype('uint8')).save(folder / f'{index:04d}.png')
    return root / 'train' / 'HR', root / 'valid' / 'HR'

if IN_KAGGLE:
    import kagglehub
    dataset_root = Path(kagglehub.dataset_download(DATASET_HANDLE))
    train_hr = find_hr_split(dataset_root, 'train')
    val_hr = find_hr_split(dataset_root, 'valid')
else:
    train_hr, val_hr = make_local_sources(OUTPUT_DIR / 'synthetic-source')

print('train:', train_hr, len(image_files(train_hr)))
print('valid:', val_hr, len(image_files(val_hr)))

In [ ]:
raw_paths = image_files(train_hr)[:6]
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, path in zip(axes.flat, raw_paths):
    ax.imshow(Image.open(path).convert('RGB')); ax.set_title(path.name); ax.axis('off')
fig.suptitle('DIV2K HR source samples'); fig.tight_layout()
fig.savefig(OUTPUT_DIR / '00_source_samples.png', dpi=150, bbox_inches='tight'); plt.show()

## 2. VP schedule과 Gaussian-mixture target 시각화
동일한 원본에서 VP level 변화와 local·edge·high-frequency·anisotropic Gaussian component를 비교합니다. Goal은 이후 blur/downsample된 dream 형태로 저장됩니다.

In [ ]:
from torchvision import transforms
from stochastic_bridge.schedule import VPNoiseSchedule

runtime_config = OUTPUT_DIR / 'runtime_config.yaml'
import yaml
cfg = yaml.safe_load((REPO_DIR / 'configs/kaggle_div2k_fullres_axial.yaml').read_text())
cfg['train']['epochs'] = EPOCHS
cfg['train']['output_dir'] = str(OUTPUT_DIR)
cfg['loss']['spatial_ce_weight'] = 0.0  # Energy-only: CE를 명시적으로 비활성화
cfg['data']['batch_size'] = 2
assert cfg['loss']['name'] == 'energy_full_band' and cfg['loss']['spatial_ce_weight'] == 0.0
if LOCAL_SMOKE:
    cfg['data'].update(image_size=32, batch_size=1, workers=0)
    cfg['prepare'].update(variants_per_image=1, batch_size=4, workers=0, shard_size=8)
    cfg['model'].update(heads=4, fullres_dim=32, fullres_depth=2, fullres_cross_depth=1, fullres_window_size=4, fullres_gradient_checkpointing=False)
    cfg['loss']['samples'] = 2
runtime_config.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
IMAGE_SIZE = cfg['data']['image_size']
transform = transforms.Compose([transforms.Resize(IMAGE_SIZE, antialias=True), transforms.CenterCrop(IMAGE_SIZE), transforms.ToTensor(), transforms.Lambda(lambda x: x * 2 - 1)])
clean = transform(Image.open(raw_paths[0]).convert('RGB'))[None]
schedule = VPNoiseSchedule(cfg['schedule']['steps'], cfg['schedule']['beta_start'], cfg['schedule']['beta_end'])
levels = [0, 10, 25, 50, 75, 100]
fixed_noise = torch.randn_like(clean)
noisy = [schedule.q_sample(clean, torch.tensor([level]), fixed_noise)[0] for level in levels]
grid = make_grid([(x.clamp(-1,1)+1)/2 for x in noisy], nrow=len(levels), padding=2)
plt.figure(figsize=(15,3)); plt.imshow(grid.permute(1,2,0)); plt.axis('off'); plt.title(' | '.join(f'level {x}' for x in levels))
plt.savefig(OUTPUT_DIR / '01_gaussian_levels.png', dpi=160, bbox_inches='tight'); plt.show()
snr = schedule.alpha_bars / (1 - schedule.alpha_bars).clamp_min(1e-10)
plt.figure(figsize=(7,4)); plt.semilogy(snr.numpy()); plt.xlabel('corruption level'); plt.ylabel('SNR'); plt.grid(alpha=.3); plt.title('VP Gaussian schedule')
plt.savefig(OUTPUT_DIR / '02_schedule_snr.png', dpi=150, bbox_inches='tight'); plt.show()

from stochastic_bridge.noise import CorruptionMixture
mixture_cfg = dict(cfg['corruption']); mixture_cfg.pop('enabled')
gaussian_mixture = CorruptionMixture(schedule, **mixture_cfg)
gaussian_names = ['local_gaussian', 'edge_gaussian', 'high_frequency_gaussian', 'anisotropic_gaussian']
gaussian_examples = [clean[0]] + [gaussian_mixture.corrupt(clean, torch.tensor([75]), (name,))[0] for name in gaussian_names]
fig, axes = plt.subplots(1, len(gaussian_examples), figsize=(16, 3.4))
for ax, image, title in zip(axes, gaussian_examples, ['clean', *gaussian_names]):
    ax.imshow(((image.permute(1,2,0).clamp(-1,1)+1)/2).numpy()); ax.set_title(title); ax.axis('off')
fig.tight_layout(); fig.savefig(OUTPUT_DIR / '02b_gaussian_mixture.png', dpi=160, bbox_inches='tight'); plt.show()

## 3. 고정 Gaussian-mixture bridge dataset 생성
Current·answer cloud는 Gaussian component 혼합에서 생성하고 goal에는 추가 blur/downsample을 적용해 dream처럼 만듭니다. 학습과 validation은 원본 이미지 기준으로 분리하며 prepared data는 `/kaggle/temp`에 저장합니다. 기본값은 3,200 train records로 시작하며 상단의 `TRAIN_VARIANTS`를 늘리면 데이터량도 선형 증가합니다.

In [ ]:
train_prepared = CACHE_DIR / 'train'
val_prepared = CACHE_DIR / 'valid'

def prepare_if_needed(source, destination, variants):
    manifest = destination / 'manifest.json'
    if FORCE_REBUILD_DATA and destination.exists(): shutil.rmtree(destination)
    if manifest.exists():
        print('reuse:', manifest); return
    command = [sys.executable, 'prepare_dataset.py', '--config', str(runtime_config), '--data', str(source), '--output', str(destination), '--variants', str(variants), '--device', 'cuda' if torch.cuda.is_available() else 'cpu']
    subprocess.run(command, check=True)

prepare_if_needed(train_hr, train_prepared, TRAIN_VARIANTS)
prepare_if_needed(val_hr, val_prepared, VAL_VARIANTS)
train_manifest = json.loads((train_prepared / 'manifest.json').read_text())
val_manifest = json.loads((val_prepared / 'manifest.json').read_text())
print('train records:', train_manifest['length'], 'shards:', len(train_manifest['shards']))
print('valid records:', val_manifest['length'], 'shards:', len(val_manifest['shards']))
print('analytic VP:', train_manifest['analytic_vp'], 'stored target noise:', train_manifest['has_target_noise'])

from collections import Counter
def prepared_stats(root, manifest):
    counts, endpoints, total = Counter(), 0, 0
    names = manifest['corruption_names']
    for shard in manifest['shards']:
        payload = torch.load(root / shard['file'], map_location='cpu', weights_only=True, mmap=True)
        endpoints += int((payload['answer_level'] == 0).sum()); total += len(payload['answer_level'])
        counts.update(names[int(i)] if int(i) >= 0 else 'none' for i in payload['corruption_type_id'])
    return {'records': total, 'clean_endpoint_fraction': endpoints / total, 'corruptions': dict(counts)}
train_stats = prepared_stats(train_prepared, train_manifest)
val_stats = prepared_stats(val_prepared, val_manifest)
display(pd.DataFrame([train_stats, val_stats], index=['train', 'valid']))
assert train_manifest['analytic_vp'] is False and train_manifest['has_target_noise'] is False
assert 0.0 < train_stats['clean_endpoint_fraction'] <= 1.0
assert sum(count for name, count in train_stats['corruptions'].items() if name != 'white_gaussian') > 0

In [ ]:
from stochastic_bridge.prepared import PreparedBridgeDataset
fixed_train = PreparedBridgeDataset(train_prepared, cache_shards=2)
white_id = fixed_train.manifest['corruption_names'].index('white_gaussian')
sample_index = next(i for i in range(min(len(fixed_train), 256)) if int(fixed_train[i]['corruption_type_id']) != white_id)
sample = fixed_train[sample_index]
corruption_name = fixed_train.manifest['corruption_names'][int(sample['corruption_type_id'])]
panels = [sample['clean'], sample['current'], sample['goal'], *list(sample['target_cloud'][:min(4, len(sample['target_cloud']))])]
labels = ['clean', f"current s={sample['current_level'].item()}", f"goal r={sample['goal_level'].item()}"] + [f'target {i}' for i in range(len(panels)-3)]
grid = make_grid([(x.clamp(-1,1)+1)/2 for x in panels], nrow=len(panels), padding=2)
plt.figure(figsize=(16,3)); plt.imshow(grid.permute(1,2,0)); plt.axis('off'); plt.title(corruption_name + ' | ' + ' | '.join(labels) + f" | answer a={sample['answer_level'].item()}")
plt.savefig(OUTPUT_DIR / '03_fixed_bridge_record.png', dpi=160, bbox_inches='tight'); plt.show()

## 4. T4×2 DDP 학습
두 GPU에서는 `torchrun --nproc_per_node=2`와 NCCL을 사용합니다. native 128 기준 batch size는 **GPU당 2**, global batch는 4이며 각 condition에서 4개 cloud sample을 생성합니다. `spatial_ce_weight=0`이므로 loss는 오직 full-band Energy distance입니다. rank 0만 validation·history·checkpoint를 저장하며 중단 후 `latest.pt`에서 자동 resume합니다.

In [ ]:
ddp_script = REPO_DIR / 'kaggle/train_div2k_ddp.py'
if not ddp_script.exists(): raise FileNotFoundError('kaggle/train_div2k_ddp.py가 없습니다. 최신 저장소를 push/clone했는지 확인하세요.')
resume_path = OUTPUT_DIR / 'latest.pt'
common = ['--config', str(runtime_config), '--train-prepared', str(train_prepared), '--val-prepared', str(val_prepared), '--output', str(OUTPUT_DIR), '--epochs', str(EPOCHS)]
if RESUME_IF_AVAILABLE and resume_path.exists():
    resume_state = torch.load(resume_path, map_location='cpu', weights_only=False)
    has_fullres_axial = any(key.startswith('fullres.cross_blocks.') for key in resume_state['model'])
    if has_fullres_axial and resume_state.get('config', {}).get('loss', {}).get('spatial_ce_weight', 0.0) == 0.0:
        common += ['--resume', str(resume_path)]
    else:
        print('기존 checkpoint가 fullres axial Energy-only 구조와 달라 새로 학습합니다.')
elif INIT_CHECKPOINT:
    init_path = Path(INIT_CHECKPOINT)
    if not init_path.is_file(): raise FileNotFoundError(f'INIT_CHECKPOINT not found: {init_path}')
    common += ['--init-checkpoint', str(init_path)]
if LOCAL_SMOKE: common += ['--max-train-batches', '2', '--max-val-batches', '1']
gpu_count = torch.cuda.device_count()
if gpu_count >= 2:
    command = ['torchrun', '--standalone', '--nproc_per_node=2', str(ddp_script), *common]
else:
    command = [sys.executable, str(ddp_script), *common]
print(' '.join(command)); subprocess.run(command, cwd=REPO_DIR, check=True)

## 5. 학습 곡선
복원 PSNR은 ensemble mean과 clean image 사이의 값입니다. 현재 단계가 완전 clean answer를 뜻하지 않는 경우에도 일관된 비교 지표로 사용합니다.

In [ ]:
history = pd.DataFrame(json.loads((OUTPUT_DIR / 'history.json').read_text()))
required_lambda_columns = {'train_noise_variance', 'val_noise_variance', 'train_cloud_loss', 'val_cloud_loss', 'train_spatial_ce', 'val_spatial_ce'}
missing_lambda_columns = required_lambda_columns.difference(history.columns)
if missing_lambda_columns:
    raise RuntimeError(f'구형 trainer의 history.json입니다 (누락: {sorted(missing_lambda_columns)}). 최신 코드를 push한 뒤 Section 4 학습 셀부터 다시 실행하세요.')
if history[['train_spatial_ce','val_spatial_ce']].abs().to_numpy().max() != 0:
    raise RuntimeError('CE가 활성화된 history입니다. spatial_ce_weight=0 설정을 확인하세요.')
display(history)
fig, axes = plt.subplots(2, 3, figsize=(17, 8))
axes[0,0].plot(history.epoch, history.train_cloud_loss, label='train'); axes[0,0].plot(history.epoch, history.val_cloud_loss, label='valid'); axes[0,0].set_title('Full-band Energy distance (CE=0)'); axes[0,0].legend()
axes[0,1].plot(history.epoch, history.val_current_psnr, label='current input'); axes[0,1].plot(history.epoch, history.val_output_psnr, label='reconstructed mean'); axes[0,1].set_title('Validation PSNR to clean'); axes[0,1].legend()
axes[0,2].plot(history.epoch, history.train_noise_variance, label='train'); axes[0,2].plot(history.epoch, history.val_noise_variance, label='valid'); axes[0,2].set_title('Mean learned spatial energy'); axes[0,2].legend()
axes[1,0].plot(history.epoch, history.gradient_norm); axes[1,0].set_title('Gradient norm')
axes[1,1].plot(history.epoch, history.learning_rate); axes[1,1].set_title('Learning rate')
axes[1,2].plot(history.epoch, history.val_output_psnr-history.val_current_psnr); axes[1,2].axhline(0, color='black', lw=1); axes[1,2].set_title('Validation ΔPSNR')
for ax in axes.flat: ax.grid(alpha=.3); ax.set_xlabel('epoch')
fig.tight_layout(); fig.savefig(OUTPUT_DIR / '04_training_curves.png', dpi=160, bbox_inches='tight'); plt.show()
history.to_csv(OUTPUT_DIR / 'history.csv', index=False)

## 6. checkpoint 로드와 corruption level별 복원력
각 level에서 current는 `s`, goal은 `max(s−20,0)`, 요청 answer는 `max(goal−10,0)`입니다. current PSNR, ensemble-mean 복원 PSNR, target/predicted diversity를 같이 측정합니다.

In [ ]:
from stochastic_bridge.config import config_from_dict
from stochastic_bridge.model import StochasticImageBridge

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
checkpoint = torch.load(OUTPUT_DIR / 'best.pt', map_location=device, weights_only=False)
loaded_cfg = config_from_dict(checkpoint['config'])
model = StochasticImageBridge(**loaded_cfg.model.__dict__).to(device)
model.load_state_dict(checkpoint['model']); model.eval()
schedule = VPNoiseSchedule(loaded_cfg.schedule.steps, loaded_cfg.schedule.beta_start, loaded_cfg.schedule.beta_end).to(device)
clean_eval = PreparedBridgeDataset(val_prepared)[0]['clean'][None].to(device)
from stochastic_bridge.corruptions import GoalDetailCorruptor
dream_cfg = loaded_cfg.goal_corruption.__dict__.copy(); dream_cfg.pop('enabled')
dream_corruptor = GoalDetailCorruptor(**dream_cfg).to(device)
assert loaded_cfg.loss.spatial_ce_weight == 0.0 and loaded_cfg.loss.name == 'energy_full_band'

def image_psnr(a, b):
    mse = (a-b).square().flatten(1).mean(1).clamp_min(1e-10)
    return 10 * torch.log10(4 / mse)

@torch.inference_mode()
def evaluate_level(level, samples=EVAL_SAMPLES):
    s = torch.tensor([level], device=device, dtype=torch.long)
    r = torch.tensor([max(level-20, 0)], device=device, dtype=torch.long)
    a = (r - loaded_cfg.schedule.answer_jump).clamp_min(0)
    current = schedule.q_sample(clean_eval, s)
    goal = dream_corruptor(schedule.q_sample(clean_eval, r))
    target_noise = torch.randn(1, samples, *clean_eval.shape[1:], device=device)
    target = schedule.sample_answer_cloud(clean_eval, current, s, a, samples, target_noise)
    predicted, learned_variance, spatial_energy = model(current, goal, samples=samples, noise=None, return_noise_variance=True, return_noise_energy=True)
    return {'level': level, 's': s, 'r': r, 'a': a, 'current': current, 'goal': goal, 'target': target, 'predicted': predicted, 'learned_variance': learned_variance, 'spatial_energy': spatial_energy, 'pred_mean': predicted.mean(1), 'target_mean': target.mean(1)}

levels = [10, 25, 40, 60, 80, 100]
results = [evaluate_level(level) for level in levels]
rows = []
for result in results:
    rows.append({'input_level': result['level'], 'goal_level': result['r'].item(), 'answer_level': result['a'].item(), 'learned_noise_variance': result['learned_variance'].item(), 'current_psnr': image_psnr(result['current'], clean_eval).item(), 'output_mean_psnr': image_psnr(result['pred_mean'], clean_eval).item(), 'target_mean_psnr': image_psnr(result['target_mean'], clean_eval).item(), 'predicted_diversity': result['predicted'].std(1).mean().item(), 'target_diversity': result['target'].std(1).mean().item()})
severity = pd.DataFrame(rows); display(severity); severity.to_csv(OUTPUT_DIR / 'severity_metrics.csv', index=False)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17,4))
axes[0].plot(severity.input_level, severity.current_psnr, 'o-', label='current → clean')
axes[0].plot(severity.input_level, severity.output_mean_psnr, 'o-', label='output mean → clean')
axes[0].plot(severity.input_level, severity.target_mean_psnr, 'o--', label='target mean → clean')
axes[0].set_ylabel('PSNR (dB)'); axes[0].legend(); axes[0].set_title('Restoration by corruption level')
axes[1].plot(severity.input_level, severity.predicted_diversity, 'o-', label='predicted cloud')
axes[1].plot(severity.input_level, severity.target_diversity, 'o-', label='target cloud')
axes[1].set_ylabel('mean pixel std'); axes[1].legend(); axes[1].set_title('Distribution spread')
axes[2].plot(severity.input_level, severity.learned_noise_variance, 'o-', color='tab:purple')
axes[2].set_ylabel('mean w'); axes[2].set_title('Mean spatial noise energy')
for ax in axes: ax.set_xlabel('input corruption level'); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(OUTPUT_DIR / '05_severity_curves.png', dpi=160, bbox_inches='tight'); plt.show()

fig, axes = plt.subplots(len(results), 7, figsize=(15, 2.2*len(results)))
for row, result in enumerate(results):
    images = [clean_eval[0], result['current'][0], result['goal'][0], result['target_mean'][0], result['pred_mean'][0], (result['pred_mean'][0]-result['target_mean'][0]).abs()*3-1, result['predicted'][0].std(0)*6-1]
    titles = ['clean','current','goal','target mean','pred mean','|mean error| ×3','pred std ×6']
    for col, (image, title) in enumerate(zip(images, titles)):
        axes[row,col].imshow(((image.clamp(-1,1)+1)/2).permute(1,2,0).cpu()); axes[row,col].axis('off')
        if row == 0: axes[row,col].set_title(title)
    axes[row,0].set_ylabel(f"s={result['level']}")
fig.tight_layout(); fig.savefig(OUTPUT_DIR / '06_severity_reconstructions.png', dpi=160, bbox_inches='tight'); plt.show()

### Gaussian component별 clean 복원력
각 Gaussian family를 current level 75, goal level 50으로 적용하고 dream goal 조건에서 clean `x₀` 복원 성능, 출력 다양성, 평균 spatial energy를 비교합니다.

In [ ]:
corruption = loaded_cfg.corruption
structured_mixture = CorruptionMixture(schedule, corruption.weights, corruption.spatial_floor, corruption.student_t_df).to(device)
structured_types = ['white_gaussian', 'local_gaussian', 'edge_gaussian', 'low_frequency_gaussian', 'high_frequency_gaussian', 'band_pass_gaussian', 'channel_correlated_gaussian', 'low_rank_gaussian', 'anisotropic_gaussian', 'signal_dependent_gaussian']
structured_rows, structured_images = [], []
with torch.inference_mode():
    for name in structured_types:
        current = structured_mixture.corrupt(clean_eval, torch.tensor([75], device=device), (name,))
        goal = dream_corruptor(structured_mixture.corrupt(clean_eval, torch.tensor([50], device=device), (name,)))
        cloud, variance, energy_map = model(current, goal, samples=EVAL_SAMPLES, noise=None, return_noise_variance=True, return_noise_energy=True)
        mean, std = cloud.mean(1), cloud.std(1)
        structured_rows.append({'corruption': name, 'current_psnr': image_psnr(current, clean_eval).item(), 'output_psnr': image_psnr(mean, clean_eval).item(), 'delta_psnr': (image_psnr(mean, clean_eval)-image_psnr(current, clean_eval)).item(), 'mean_spatial_energy': variance.item(), 'max_spatial_energy': energy_map.max().item(), 'output_diversity': std.mean().item()})
        structured_images.append((current[0].cpu(), goal[0].cpu(), mean[0].cpu(), std[0].cpu()))
structured_metrics = pd.DataFrame(structured_rows); display(structured_metrics)
structured_metrics.to_csv(OUTPUT_DIR / 'structured_corruption_metrics.csv', index=False)
fig, axes = plt.subplots(len(structured_types), 5, figsize=(12, 2.25*len(structured_types)))
for row, (name, images) in enumerate(zip(structured_types, structured_images)):
    current, goal, mean, std = images
    panels = [clean_eval[0].cpu(), current, goal, mean, std*6-1]
    for col, (image, title) in enumerate(zip(panels, ['clean','current','goal','output mean','output std ×6'])):
        axes[row,col].imshow(((image.clamp(-1,1)+1)/2).permute(1,2,0)); axes[row,col].axis('off')
        if row == 0: axes[row,col].set_title(title)
    axes[row,0].set_ylabel(name)
fig.tight_layout(); fig.savefig(OUTPUT_DIR / '06b_structured_reconstructions.png', dpi=160, bbox_inches='tight'); plt.show()

## 7. 실제 반복 rollout
Blur/downsample된 dream `goal₀`를 모든 step에서 고정하고 모델 출력을 다음 current로 다시 입력합니다. 주 용도는 1~2-step 생성이지만 8회까지 기록해 이후 drift도 확인하며, 각 trajectory의 full-resolution latent를 고정합니다.

In [ ]:
import torch.nn.functional as F

def batch_ssim(x, y):
    x, y = (x.float()+1)/2, (y.float()+1)/2
    mu_x, mu_y = F.avg_pool2d(x, 7, 1, 3), F.avg_pool2d(y, 7, 1, 3)
    var_x = F.avg_pool2d(x*x, 7, 1, 3) - mu_x.square()
    var_y = F.avg_pool2d(y*y, 7, 1, 3) - mu_y.square()
    covariance = F.avg_pool2d(x*y, 7, 1, 3) - mu_x*mu_y
    score = ((2*mu_x*mu_y + .01**2) * (2*covariance + .03**2)) / ((mu_x.square()+mu_y.square()+.01**2) * (var_x+var_y+.03**2)).clamp_min(1e-8)
    return score.flatten(1).mean(1)

def batch_cosine(x, y):
    return F.cosine_similarity(x.flatten(1).float(), y.flatten(1).float(), dim=1)

@torch.inference_mode()
def fixed_goal_rollout(paths=4, iterations=8):
    target = clean_eval.expand(paths, -1, -1, -1)
    goal_0 = dream_corruptor(target)
    initial_noise = torch.randn_like(target[:1]).expand_as(target)
    current = schedule.q_sample(target, torch.full((paths,), 100, device=device, dtype=torch.long), initial_noise)
    persistent_latent = torch.randn(paths, 1, model.fullres.dim, *target.shape[-2:], device=device)
    target_h, _ = model.encode_condition(goal_0, goal_0)
    states, metric_rows, psnr_paths, ssim_paths = [], [], [], []
    previous = None
    for step in range(iterations + 1):
        h_t, _ = model.encode_condition(current, goal_0)
        psnr_values = image_psnr(current.float(), target.float())
        ssim_values = batch_ssim(current, target)
        image_cosine = batch_cosine(current, target)
        hidden_cosine = batch_cosine(h_t, target_h)
        learned_variance = model.noise_variance_from_condition(h_t).squeeze(-1)
        update = torch.zeros(paths, device=device) if previous is None else (current-previous).abs().flatten(1).mean(1)
        metric_rows.append({'step': step, 'psnr_mean': psnr_values.mean().item(), 'psnr_std': psnr_values.std().item(), 'ssim_mean': ssim_values.mean().item(), 'ssim_std': ssim_values.std().item(), 'image_cosine_mean': image_cosine.mean().item(), 'image_l1_mean': (current-target).abs().mean().item(), 'h_goal_cosine_mean': hidden_cosine.mean().item(), 'h_goal_cosine_std': hidden_cosine.std().item(), 'learned_noise_variance_mean': learned_variance.mean().item(), 'learned_noise_variance_std': learned_variance.std().item(), 'update_l1_mean': update.mean().item(), 'trajectory_std': current.std(0).mean().item()})
        psnr_paths.append(psnr_values.cpu()); ssim_paths.append(ssim_values.cpu()); states.append(current.detach().cpu())
        if step == iterations: break
        previous = current
        current = model(current, goal_0, samples=1, noise=persistent_latent)[:, 0]
    return {'states': states, 'goal': goal_0[0].detach().cpu(), 'metrics': pd.DataFrame(metric_rows), 'psnr': torch.stack(psnr_paths, 1).numpy(), 'ssim': torch.stack(ssim_paths, 1).numpy()}

rollout = fixed_goal_rollout(paths=4 if not LOCAL_SMOKE else 2, iterations=8)
rollout_metrics = rollout['metrics']; rollout_metrics.to_csv(OUTPUT_DIR / 'rollout_metrics.csv', index=False); display(rollout_metrics.iloc[::5])
rollout_psnr = rollout['psnr']; selected_steps = [0,1,2,3,4,6,8]
display_paths = min(4, rollout['states'][0].shape[0])
fig, axes = plt.subplots(display_paths, len(selected_steps), figsize=(16, 2.3*display_paths), squeeze=False)
for row in range(display_paths):
    for col, step in enumerate(selected_steps):
        state = rollout['states'][step][row]
        axes[row,col].imshow(((state.clamp(-1,1)+1)/2).permute(1,2,0)); axes[row,col].axis('off')
        if row == 0: axes[row,col].set_title(f'step {step}')
    axes[row,0].set_ylabel(f'trajectory {row}')
fig.suptitle('Fixed goal₀ recurrent rollout — persistent latent per trajectory', y=1.01)
fig.tight_layout(); fig.savefig(OUTPUT_DIR / '10_rollout_trajectories.png', dpi=160, bbox_inches='tight'); plt.show()

fig, axes = plt.subplots(2,3,figsize=(17,8)); steps = rollout_metrics.step
for values in rollout_psnr: axes[0,0].plot(steps, values, color='tab:blue', alpha=.2)
axes[0,0].plot(steps, rollout_metrics.psnr_mean, color='black', label='mean'); axes[0,0].fill_between(steps, rollout_metrics.psnr_mean-rollout_metrics.psnr_std, rollout_metrics.psnr_mean+rollout_metrics.psnr_std, alpha=.2); axes[0,0].set_title('PSNR to fixed goal₀')
axes[0,1].plot(steps, rollout_metrics.ssim_mean, label='SSIM'); axes[0,1].plot(steps, rollout_metrics.image_cosine_mean, label='image cosine'); axes[0,1].set_title('Image similarity'); axes[0,1].legend()
axes[0,2].plot(steps, rollout_metrics.learned_noise_variance_mean, color='tab:red'); axes[0,2].fill_between(steps, rollout_metrics.learned_noise_variance_mean-rollout_metrics.learned_noise_variance_std, rollout_metrics.learned_noise_variance_mean+rollout_metrics.learned_noise_variance_std, alpha=.2, color='tab:red'); axes[0,2].set_title('Mean spatial noise energy')
axes[1,0].plot(steps, rollout_metrics.h_goal_cosine_mean, color='tab:purple'); axes[1,0].fill_between(steps, rollout_metrics.h_goal_cosine_mean-rollout_metrics.h_goal_cosine_std, rollout_metrics.h_goal_cosine_mean+rollout_metrics.h_goal_cosine_std, alpha=.2, color='tab:purple'); axes[1,0].set_title('cosine(h_t, h_goal₀)')
axes[1,1].semilogy(steps[1:], rollout_metrics.update_l1_mean.iloc[1:], label='|x_t−x_{t−1}|'); axes[1,1].semilogy(steps, rollout_metrics.trajectory_std.clip(lower=1e-8), label='trajectory std'); axes[1,1].set_title('Convergence / diversity'); axes[1,1].legend()
axes[1,2].plot(steps, rollout_metrics.image_l1_mean, label='image L1'); axes[1,2].plot(steps[1:], rollout_metrics.update_l1_mean.iloc[1:], label='update L1'); axes[1,2].set_title('Error / update'); axes[1,2].legend()
for ax in axes.flat: ax.set_xlabel('recurrent step'); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(OUTPUT_DIR / '11_rollout_metrics.png', dpi=160, bbox_inches='tight'); plt.show()

final_cloud = rollout['states'][-1]; final_mean, final_std = final_cloud.mean(0), final_cloud.std(0).mean(0)
fig, axes = plt.subplots(1,5,figsize=(15,3)); rollout_maps = [clean_eval[0].cpu(), rollout['goal'], final_cloud[0], final_mean, final_std]; rollout_titles = ['sharp target','dream goal₀','one final sample','final ensemble mean','final ensemble std']
for ax, value, title in zip(axes, rollout_maps, rollout_titles):
    if value.ndim == 3: ax.imshow(((value.clamp(-1,1)+1)/2).permute(1,2,0))
    else: ax.imshow(value, cmap='magma')
    ax.set_title(title); ax.axis('off')
fig.tight_layout(); fig.savefig(OUTPUT_DIR / '12_rollout_endpoint.png', dpi=160, bbox_inches='tight'); plt.show()
print(f"final-step PSNR: {rollout_psnr[:,-1].mean():.2f} ± {rollout_psnr[:,-1].std():.2f} dB | h-goal cosine: {rollout_metrics.h_goal_cosine_mean.iloc[-1]:.4f}")

## 8. 생성분포 분석
한 condition에서 target cloud와 model cloud를 correction-feature PCA로 비교합니다. 이어서 target variance가 가장 큰 픽셀의 분포와 공간 uncertainty map을 확인합니다. 완전히 겹치면 분포 matching이 잘 된 것이고, 평균만 맞고 폭이 다르면 diversity calibration 문제가 남은 것입니다.

In [ ]:
from stochastic_bridge.losses import LaplacianCorrectionFeatures
distribution = evaluate_level(80, samples=max(EVAL_SAMPLES, 32))
target = distribution['target'][0].float(); predicted = distribution['predicted'][0].float(); current = distribution['current'].float()
features = LaplacianCorrectionFeatures(levels=loaded_cfg.loss.full_band_levels).to(device)
with torch.no_grad():
    current_feature = features(current)
    target_feature = features(target) - current_feature
    predicted_feature = features(predicted) - current_feature
    joined = torch.cat([target_feature, predicted_feature]); joined = joined - joined.mean(0, keepdim=True)
    _, _, basis = torch.pca_lowrank(joined, q=2); points = (joined @ basis).cpu().numpy()
n = len(target_feature)
plt.figure(figsize=(6,5)); plt.scatter(points[:n,0], points[:n,1], alpha=.65, label='target'); plt.scatter(points[n:,0], points[n:,1], alpha=.65, label='predicted'); plt.xlabel('PC1'); plt.ylabel('PC2'); plt.title('Correction-cloud PCA at input level 80'); plt.legend(); plt.grid(alpha=.25)
plt.savefig(OUTPUT_DIR / '07_cloud_pca.png', dpi=160, bbox_inches='tight'); plt.show()

target_std = target.std(0).mean(0); y, x = np.unravel_index(target_std.argmax().item(), target_std.shape)
target_pixel = target[:,:,y,x].mean(1).cpu().numpy(); predicted_pixel = predicted[:,:,y,x].mean(1).cpu().numpy()
plt.figure(figsize=(7,4)); plt.hist(target_pixel, bins=15, alpha=.6, density=True, label='target'); plt.hist(predicted_pixel, bins=15, alpha=.6, density=True, label='predicted'); plt.title(f'High-variance pixel distribution at (y={y}, x={x})'); plt.xlabel('mean RGB value [-1,1]'); plt.legend()
plt.savefig(OUTPUT_DIR / '08_pixel_distribution.png', dpi=160, bbox_inches='tight'); plt.show()

In [ ]:
pred_mean = predicted.mean(0); target_mean = target.mean(0)
pred_std = predicted.std(0).mean(0); target_std = target.std(0).mean(0)
mean_error = (pred_mean-target_mean).abs().mean(0)
spatial_energy = distribution['spatial_energy'][0].float()
maps = [pred_mean, target_mean, pred_std, target_std, mean_error, spatial_energy]
titles = ['predicted mean','target mean','predicted std','target std','mean absolute error','learned spatial energy w']
fig, axes = plt.subplots(1,6,figsize=(19,3.4))
for ax, value, title in zip(axes, maps, titles):
    if value.ndim == 3: ax.imshow(((value.clamp(-1,1)+1)/2).permute(1,2,0).cpu())
    else: ax.imshow(value.cpu(), cmap='magma');
    ax.set_title(title); ax.axis('off')
fig.tight_layout(); fig.savefig(OUTPUT_DIR / '09_uncertainty_maps.png', dpi=160, bbox_inches='tight'); plt.show()

best_rollout_index = int(rollout_metrics.psnr_mean.idxmax())
summary = {'loss': 'energy_full_band_only', 'spatial_ce_weight': 0.0, 'best_epoch': int(history.loc[history.val_loss.idxmin(), 'epoch']), 'best_val_loss': float(history.val_loss.min()), 'final_val_output_psnr': float(history.val_output_psnr.iloc[-1]), 'final_mean_spatial_energy': float(history.val_noise_variance.iloc[-1]), 'level80_predicted_diversity': float(predicted.std(0).mean()), 'level80_target_diversity': float(target.std(0).mean()), 'level80_mean_spatial_energy': float(distribution['learned_variance'].item()), 'level80_max_spatial_energy': float(distribution['spatial_energy'].max()), 'rollout_best_step': int(rollout_metrics.loc[best_rollout_index, 'step']), 'rollout_best_psnr': float(rollout_metrics.loc[best_rollout_index, 'psnr_mean']), 'rollout_final_psnr_mean': float(rollout_psnr[:,-1].mean()), 'rollout_final_psnr_std': float(rollout_psnr[:,-1].std()), 'rollout_final_h_goal_cosine': float(rollout_metrics.h_goal_cosine_mean.iloc[-1]), 'rollout_final_mean_spatial_energy': float(rollout_metrics.learned_noise_variance_mean.iloc[-1]), 'rollout_final_update_l1': float(rollout_metrics.update_l1_mean.iloc[-1])}
(OUTPUT_DIR / 'analysis_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))

## 해석 체크리스트

- `train_spatial_ce`와 `val_spatial_ce`가 정확히 0인지 먼저 확인한다. 이 실험에는 CE가 없다.
- `val_loss`가 감소하면서 `val_output_psnr`가 상승하는가?
- 높은 corruption level에서도 output PSNR이 current PSNR보다 높은가?
- PCA에서 predicted cloud가 target cloud의 중심뿐 아니라 폭과 방향도 따라가는가?
- predicted std가 target std보다 지속적으로 작으면 latent collapse 가능성이 있다.
- predicted std가 과도하게 크면 noise injection 또는 residual gate가 불안정할 수 있다.
- spatial energy가 CE 없이도 edge·texture·실제 훼손 위치에 집중되는지 확인한다. 평탄 영역 전체에 균일하면 Energy gradient만으로 위치 선택을 배우지 못한 것이다.
- one-step PSNR은 좋은데 rollout PSNR이 후반에 무너지면 exposure bias 또는 off-manifold 누적 오차가 핵심 문제다.

Kaggle에서 **Save Version → Save & Run All**을 사용하면 `/kaggle/working`의 checkpoint와 figure가 notebook output으로 보존됩니다.

In [ ]:
from IPython.display import FileLink, display
print('Saved artifacts:')
for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file(): print(f'{path.name:32s} {path.stat().st_size/2**20:8.2f} MiB')
archive_base = OUTPUT_DIR.parent / f'{OUTPUT_DIR.name}_artifacts'
archive = shutil.make_archive(str(archive_base), 'zip', OUTPUT_DIR, '.', logger=None)
print('archive:', archive)
if IN_KAGGLE: display(FileLink(archive))